# 🔍 Analisis Exploratorio y Limpieza — INUMET
**Materia:** Herramientas de Software para Big Data  
**Fuente:** Instituto Uruguayo de Meteorologia  

**Flujo de zonas:**
```
/lnd  →  datos crudos tal cual llegan de NiFi
/raw  →  esquema corregido y tipos validados
/rfn  →  datos limpios: nulos, duplicados y reglas de negocio aplicadas
```

**Tablas:**
| Archivo | Columnas |
|---------|----------|
| inumet_temperatura_del_aire.csv | fecha, estacion_id, temp_aire (°C) |
| inumet_intensidad_de_viento.csv | fecha, estacion_id, dir_viento (°), int_viento (km/h) |
| inumet_precipitacion_acumulada_horaria.csv | fecha, estacion_id, precip_horaria (mm) |
| inumet_humedad_relativa.csv | fecha, estacion_id, [ver printSchema] |
| inumet_presion_atmosferica_a_nive_del_mar.csv | fecha, estacion_id, [ver printSchema] |
| inumet_heliofania.csv | fecha, estacion_id, [ver printSchema] |

## 1. Inicializacion de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("INUMET_EDA_Limpieza") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo")

## 2. Carga desde /lnd (datos crudos)

In [ ]:
LND = "hdfs://localhost:9000/lnd/obligatorio"
RAW = "hdfs://localhost:9000/raw/obligatorio"
RFN = "hdfs://localhost:9000/rfn/obligatorio"

col_valor_humedad = "hum_relativa"
col_valor_presion = "pres_atm_mar"
col_valor_helio   = "heliofania"

config_tablas = {
    "temperatura": {
        "archivo": "inumet_temperatura_del_aire.csv",
        "numericas": ["temp_aire"],
        "reglas": [("temp_aire", -20, 50)],
    },
    "viento": {
        "archivo": "inumet_intensidad_de_viento.csv",
        "numericas": ["dir_viento", "int_viento"],
        "reglas": [("int_viento", 0, 200), ("dir_viento", 0, 360)],
    },
    "precipitacion": {
        "archivo": "inumet_precipitacion_acumulada_horaria.csv",
        "numericas": ["precip_horario"],
        "reglas": [("precip_horario", 0, 300)],
    },
    "humedad": {
        "archivo": "inumet_humedad_relativa.csv",
        "numericas": [col_valor_humedad],
        "reglas": [(col_valor_humedad, 0, 100)],
    },
    "presion": {
        "archivo": "inumet_presion_atmosferica_a_nive_del_mar.csv",
        "numericas": [col_valor_presion],
        "reglas": [(col_valor_presion, 900, 1100)],
    },
    "heliofania": {
        "archivo": "inumet_heliofania.csv",
        "numericas": [col_valor_helio],
        "reglas": [(col_valor_helio, 0, 24)],
    },
}

In [ ]:
# Lectura cruda — sin inferSchema para ver exactamente como llegan los datos
tablas = {
    nombre: spark.read.csv(f"{LND}/{cfg['archivo']}", header=True, sep=";")
    for nombre, cfg in config_tablas.items()
}

conteos_lnd = {nombre: df.count() for nombre, df in tablas.items()}

print("Registros por tabla:")
for nombre, total in conteos_lnd.items():
    print(f"  {nombre:<15}: {total:>10,}")

## 3. Vista previa y schema de cada tabla
> En esta celda se identifican los nombres reales de columnas de humedad, presion y heliofania.

In [ ]:
for nombre, df in tablas.items():
    print(f"\n{'='*55}")
    print(f" TABLA: {nombre.upper()}")
    print(f" Columnas ({len(df.columns)}): {df.columns}")
    print(f"{'='*55}")
    df.printSchema()
    df.show(3, truncate=False)

## 4. Descripcion de columnas por tabla
Significado de cada columna dentro del dominio meteorologico.

In [ ]:
descripcion = {
    "temperatura": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        "temp_aire":  "Temperatura del aire en grados Celsius (°C)"
    },
    "viento": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        "dir_viento": "Direccion del viento en grados sexagesimales (0-360°)",
        "int_viento": "Intensidad del viento en kilometros por hora (km/h)"
    },
    "precipitacion": {
        "fecha":          "Fecha y hora de la medicion en UTC",
        "estacion_id":    "Identificador de la estacion meteorologica",
        "precip_horario": "Precipitacion acumulada en el intervalo de una hora (mm)"
    },
    "humedad": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        # Actualizar con el nombre real visto en printSchema
        "hum_relativa":      "Humedad relativa del aire expresada en porcentaje (%)"
    },
    "presion": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        # Actualizar con el nombre real visto en printSchema
        "pres_atm_mar":      "Presion atmosferica al nivel del mar en hectopascales (hPa)"
    },
    "heliofania": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        # Actualizar con el nombre real visto en printSchema
        "heliofania":      "Horas de insolacion solar registradas en el intervalo horario"
    },
}

for tabla, cols in descripcion.items():
    print(f"\n{tabla.upper()}")
    print(f"{'Columna':<20} {'Descripcion'}")
    print("-"*60)
    for col_name, desc in cols.items():
        print(f"  {col_name:<18} {desc}")

## 5. Correccion de tipos y guardado en /raw
En `/lnd` todo llega como `String`. Aqui se castean los tipos correctos y se guarda en `/raw`.

In [ ]:
def castear_tipos(df, col_numericas):
    """Castea fecha a timestamp y columnas numericas a Double."""
    columnas = []
    for c in df.columns:
        if c == "fecha":
            columnas.append(F.to_timestamp(F.col(c)).alias(c))
        elif c in col_numericas:
            columnas.append(F.col(c).cast("double").alias(c))
        else:
            columnas.append(F.col(c))
    return df.select(*columnas)

tablas_raw = {
    nombre: castear_tipos(df, config_tablas[nombre]["numericas"])
    for nombre, df in tablas.items()
}

df_temp_raw    = tablas_raw["temperatura"]
df_viento_raw  = tablas_raw["viento"]
df_lluvia_raw  = tablas_raw["precipitacion"]
df_humedad_raw = tablas_raw["humedad"]
df_presion_raw = tablas_raw["presion"]
df_helio_raw   = tablas_raw["heliofania"]

In [ ]:
print("Schemas corregidos:")
for nombre, df in tablas_raw.items():
    print(f"\n{'='*55}")
    print(f" TABLA: {nombre.upper()}")
    print(f"{'='*55}")
    df.printSchema()
    df.show(3, truncate=False)

In [ ]:
# Guardar en /raw como parquet (más eficiente que CSV para Spark)
for nombre, df in tablas_raw.items():
    df.write.mode("overwrite").parquet(f"{RAW}/{nombre}")

print("Datos guardados en /raw correctamente")

## 6. Analisis de valores nulos

In [ ]:
def reporte_nulos(df, nombre):
    total = df.count()
    nulos = df.agg(*[
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).first().asDict()
    
    print(f"\n{'='*50}")
    print(f" {nombre} — {total:,} registros")
    print(f" {'Columna':<22} {'Nulos':>8} {'%':>8}")
    print("-"*42)
    for c in df.columns:
        n = nulos[c]
        pct = n / total * 100 if total > 0 else 0
        icono = "⚠" if n > 0 else "✓"
        print(f" {icono} {c:<20} {n:>8,} {pct:>7.2f}%")

In [ ]:
for nombre, df in tablas_raw.items():
    reporte_nulos(df, nombre.upper())

## 7. Analisis de duplicados

In [ ]:
def reporte_duplicados(df, nombre, claves):
    total = df.count()
    distintos = df.dropDuplicates().count()
    duplicados = total - distintos
    claves_unicas = df.select(*claves).distinct().count()
    dup_clave = total - claves_unicas
    
    print(f"\n{nombre}")
    print(f"  Total registros:      {total:>10,}")
    print(f"  Registros unicos:     {distintos:>10,}")
    print(f"  Duplicados exactos:   {duplicados:>10,}")
    print(f"  Duplicados por clave ({claves}): {dup_clave:,}")

for nombre, df in tablas_raw.items():
    reporte_duplicados(df, nombre.upper(), ["fecha", "estacion_id"])

## 8. Claves primarias unicas
La clave primaria de cada tabla es `(fecha, estacion_id)`. Verificamos que sea unica.

In [ ]:
print("Verificacion de clave primaria (fecha + estacion_id):\n")
print(f"{'Tabla':<15} {'Total':>10} {'Claves unicas':>15} {'Unicidad'}")
print("-"*52)

for nombre, df in tablas_raw.items():
    total = df.count()
    unicos = df.select("fecha", "estacion_id").distinct().count()
    ok = "✅ OK" if total == unicos else "❌ HAY DUPLICADOS"
    print(f"{nombre:<15} {total:>10,} {unicos:>15,} {ok}")

## 9. Reglas de negocio
Rangos validos para variables meteorologicas en Uruguay segun dominio.

In [ ]:
print("Reglas de negocio definidas para el dominio meteorologico:\n")
reglas = [
    ("temp_aire",        "-20",  "50",  "Temperatura del aire en Uruguay (°C)"),
    ("int_viento",       "0",    "200", "Intensidad de viento, max historico en Uruguay (km/h)"),
    ("dir_viento",       "0",    "360", "Direccion del viento en grados sexagesimales"),
    ("precip_horario",   "0",    "300", "Precipitacion horaria, max historico en Uruguay (mm)"),
    (col_valor_humedad,  "0",    "100", "Humedad relativa, rango fisicamente posible (%)"),
    (col_valor_presion,  "900",  "1100","Presion atmosferica al nivel del mar (hPa)"),
    (col_valor_helio,    "0",    "24",  "Horas de insolacion, max posible en un dia"),
]

for columna, mn, mx, desc in reglas:
    print(f"  {columna:<25} [{mn:>4}, {mx:<4}]  {desc}")

In [ ]:
# Verificar cuantos registros violan las reglas de negocio
print("Registros fuera de rango:\n")

for tabla, cfg in config_tablas.items():
    df = tablas_raw[tabla]
    for columna, mn, mx in cfg["reglas"]:
        fuera = df.filter(
            F.col(columna).isNotNull() &
            ((F.col(columna) < mn) | (F.col(columna) > mx))
        ).count()
        icono = "⚠" if fuera > 0 else "✓"
        print(f"  {icono} {tabla:<15} {columna:<25} fuera de rango: {fuera:,}")

In [ ]:
df_helio_raw.filter(
    F.col("heliofania").isNotNull() &
    ((F.col("heliofania") < 0) | (F.col("heliofania") > 24))
).show(truncate=False)

## 10. Limpieza y guardado en /rfn
Se aplican las siguientes transformaciones:
- Eliminar registros con nulos en columnas clave
- Eliminar duplicados exactos
- Eliminar registros que violan las reglas de negocio
- Limpiar espacios en `estacion_id`

In [ ]:
def condicion_reglas(reglas):
    condicion = F.lit(True)
    for columna, mn, mx in reglas:
        condicion = condicion & (F.col(columna) >= mn) & (F.col(columna) <= mx)
    return condicion

def limpiar(df, reglas):
    """Aplica limpieza completa a un DataFrame de INUMET."""
    columnas_requeridas = ["fecha", "estacion_id"] + [c for c, _, _ in reglas]
    total_original = df.count()
    
    df = df.withColumn("estacion_id", F.trim(F.col("estacion_id")))
    df_sin_nulos = df.dropna(subset=columnas_requeridas)
    tras_nulos = df_sin_nulos.count()
    
    df_sin_duplicados = df_sin_nulos.dropDuplicates()
    tras_duplicados = df_sin_duplicados.count()
    
    df_limpio = df_sin_duplicados.filter(condicion_reglas(reglas))
    tras_reglas = df_limpio.count()
    
    print(f"  Original:              {total_original:>10,}")
    print(f"  Tras eliminar nulos:   {tras_nulos:>10,}  (-{total_original - tras_nulos:,})")
    print(f"  Tras eliminar dupl.:   {tras_duplicados:>10,}  (-{tras_nulos - tras_duplicados:,})")
    print(f"  Tras reglas negocio:   {tras_reglas:>10,}  (-{tras_duplicados - tras_reglas:,})")
    print(f"  Registros finales:     {tras_reglas:>10,}")
    
    return df_limpio

In [ ]:
tablas_rfn = {}

for nombre, df in tablas_raw.items():
    print(f"\n=== LIMPIEZA - {nombre.upper()} ===")
    tablas_rfn[nombre] = limpiar(df, config_tablas[nombre]["reglas"])

df_temp_rfn    = tablas_rfn["temperatura"]
df_viento_rfn  = tablas_rfn["viento"]
df_lluvia_rfn  = tablas_rfn["precipitacion"]
df_humedad_rfn = tablas_rfn["humedad"]
df_presion_rfn = tablas_rfn["presion"]
df_helio_rfn   = tablas_rfn["heliofania"]

In [ ]:
# Guardar en /rfn como parquet
for nombre, df in tablas_rfn.items():
    df.write.mode("overwrite").parquet(f"{RFN}/{nombre}")

print("Datos refinados guardados en /rfn correctamente")

## 11. Verificacion final — comparacion lnd vs rfn

In [ ]:
print(f"{'Tabla':<15} {'LND (crudos)':>14} {'RFN (limpios)':>14} {'Eliminados':>12}")
print("-"*58)

for nombre, df in tablas_rfn.items():
    n_lnd = conteos_lnd[nombre]
    n_rfn = df.count()
    eliminados = n_lnd - n_rfn
    pct = eliminados / n_lnd * 100 if n_lnd > 0 else 0
    print(f"{nombre:<15} {n_lnd:>14,} {n_rfn:>14,} {eliminados:>10,} ({pct:.1f}%)")

In [ ]:
spark.stop()
print("Sesion Spark cerrada.")